# Предыдущий эксперимент E1

Transformer 4/4, d=384, FFN=1536: 19 693 056 параметров.
Код модели, обучения и оценки взят из исходного проекта.
Используется тот же подготовленный BPE-кэш. Веса E1 не входят в сборку.
Запускайте отдельно от обучения V2 на CUDA GPU с BF16.


In [ ]:
import sys
import json
from pathlib import Path
import torch
from tokenizers import Tokenizer

ROOT = next(p.resolve() for p in [Path.cwd(), Path.cwd().parent]
            if (p / 'e1/hw2_train.py').is_file())
sys.path.insert(0, str(ROOT / 'e1'))
CACHE = ROOT / 'data/prepared/bpe8192'
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()
from hw2_train import train_full, make_model
from hw2_model import ModelConfig, PAD, EOS
from hw2_batches import small_split_sample
from hw2_eval_safe import run_safe, same_answer


## Полная эпоха

Фактический запуск E1 начинался с budget 8192 / max_examples 96,
после шага 500 был продолжен с budget 32768 / max_examples 256.
Сохранённые журналы содержат начало и продолжения; итоговая эпоха — 57502 шага.
Здесь используются итоговые параметры полного обучения.


In [ ]:
model = train_full(root=ROOT, stop_at=None, epochs=1, resume=True,
                   checkpoint_every=2000, validation_every=2000,
                   token_budget=32768, max_examples=256, pool_size=4096)
tok = Tokenizer.from_file(str(CACHE / 'tokenizer.json'))


## Исходная оценка на validation
Функция ниже сохранена из выполненного E1-ноутбука.

In [ ]:
import ast
import json
import time
from collections import Counter

def evaluate_generation(examples, label):
    indexed = sorted(
        enumerate(examples),
        key=lambda p: len(p[1][0])
    )

    predictions = [None] * len(examples)

    torch.cuda.synchronize()
    start_time = time.perf_counter()

    with torch.inference_mode(), torch.autocast(
        "cuda", dtype=torch.bfloat16
    ):
        for start in range(0, len(indexed), 32):
            group = indexed[start:start + 32]

            src = torch.full(
                (len(group), max(len(x[0]) for _, x in group)),
                PAD, dtype=torch.long
            )

            for j, (_, (x, y, rid)) in enumerate(group):
                src[j, :len(x)] = torch.tensor(x, dtype=torch.long)

            output = model.generate(
                src.cuda(),
                max_new_tokens=384
            ).cpu().tolist()

            for j, (original_index, _) in enumerate(group):
                predictions[original_index] = output[j]

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time

    counts = Counter()
    errors = Counter()
    details = []

    for (x, y, rid), ids in zip(examples, predictions):
        finished = EOS in ids
        tokens = ids[:ids.index(EOS)] if finished else ids

        reference = tok.decode(y[:-1], skip_special_tokens=True)
        prediction = tok.decode(tokens, skip_special_tokens=True)

        try:
            tree = ast.parse(prediction)
            syntax_ok = True
            has_solve = any(
                isinstance(node, ast.FunctionDef)
                and node.name == "solve"
                for node in tree.body
            )
        except SyntaxError:
            syntax_ok = False
            has_solve = False

        rs, rv = run_safe(reference)
        ps, pv = run_safe(prediction)

        match = (
            rs == "ok"
            and ps == "ok"
            and same_answer(rv, pv)
        )

        counts["total"] += 1
        counts["eos"] += finished
        counts["syntax_ok"] += syntax_ok
        counts["has_solve"] += has_solve
        counts["ref_supported"] += rs == "ok"
        counts["pred_supported"] += ps == "ok"
        counts["both_supported"] += rs == "ok" and ps == "ok"
        counts["matches"] += match

        if ps != "ok":
            errors[str(pv).split(":")[0]] += 1

        details.append({
            "row_id": rid,
            "question": tok.decode(x[:-1], skip_special_tokens=True),
            "reference": reference,
            "prediction": prediction,
            "reference_status": rs,
            "prediction_status": ps,
            "reference_answer": str(rv),
            "prediction_answer": str(pv),
            "match": bool(match),
            "eos": bool(finished),
            "syntax_ok": bool(syntax_ok),
        })

    n = counts["total"]

    metrics = {
        "label": label,
        "checkpoint_step": 57502,
        **dict(counts),
        "agreement_all": round(counts["matches"] / n, 4),
        "agreement_supported": round(
            counts["matches"] / max(1, counts["both_supported"]), 4
        ),
        "generation_seconds_5090": round(elapsed, 3),
        "examples_per_second": round(n / elapsed, 2),
        "prediction_errors": dict(errors),
    }

    report_dir = ROOT / "reports"
    report_dir.mkdir(parents=True, exist_ok=True)

    (report_dir / f"{label}_metrics.json").write_text(
        json.dumps(metrics, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    (report_dir / f"{label}_details.json").write_text(
        json.dumps(details, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    print(f"\n===== {label} =====")
    print(json.dumps(metrics, indent=2, ensure_ascii=False))

    print("\nFirst 3 mismatches:")
    mismatches = [r for r in details if not r["match"]]

    for r in mismatches[:3]:
        print("QUESTION:", r["question"][:250])
        print("EXPECTED:", r["reference_answer"][:100])
        print("GOT:", r["prediction_answer"][:100])
        print("---")

    return metrics


val128 = small_split_sample(
    CACHE, split="validation", count=128, per_shard=3
)

val1024 = small_split_sample(
    CACHE, split="validation", count=1024, per_shard=18
)

metrics128 = evaluate_generation(val128, "final_128")
metrics1024 = evaluate_generation(val1024, "final_1024")

print("\nCOMPARISON")
print("Step 10000: 50 / 128 = 39.06%")
print(
    "Step 57502:",
    metrics128["matches"],
    "/ 128 =",
    round(100 * metrics128["agreement_all"], 2),
    "%"
)

## Экспорт E1
Итоговый T4-ноутбук для V2 находится в `inference_colab_t4.ipynb`.

In [ ]:
import subprocess
subprocess.run([sys.executable, str(ROOT / 'e1/hw2_export_t4.py')],
               cwd=ROOT, check=True)
